In [1]:
# %%
import pandas as pd

df = pd.read_csv("/home/thinkpad/Documents/BrototypeStudying/Paper two/week 8/code/data/raw_employee_data.csv")

# %%
df = df.dropna(subset="salary")

# %%
# Keep performance_rating this time — drop only truly unneeded columns
df = df.drop(columns=["employee_id", "name", "email", "phone", "gender", "education", "join_date", "notes"])

# %%
from sklearn.model_selection import train_test_split

x = df.drop(columns=["salary"])
y = df["salary"]

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

# %%
# --- Clean age ---
median_age = x_train["age"].median()
x_train.loc[x_train["age"] > 100, "age"] = median_age
x_test.loc[x_test["age"] > 100, "age"] = median_age

x_train.loc[x_train["age"] < 18, "age"] = median_age
x_test.loc[x_test["age"] < 18, "age"] = median_age   # was missing in your version

x_train["age"] = x_train["age"].fillna(median_age)
x_test["age"] = x_test["age"].fillna(median_age)

# %%
# --- Clean department ---
department_mode = x_train["department"].mode()[0]
x_train["department"] = x_train["department"].fillna(department_mode)
x_test["department"] = x_test["department"].fillna(department_mode)

x_train["department"] = x_train["department"].str.strip().str.lower()
x_test["department"] = x_test["department"].str.strip().str.lower()

dept_map = {
    "suport": "support",
    "i.t.": "it",
    "human resources": "hr",
    "h.r.": "hr",
    "markting": "marketing",
    "ops": "operations"
}
x_train["department"] = x_train["department"].replace(dept_map)
x_test["department"] = x_test["department"].replace(dept_map)

# %%
# --- Clean city ---
city_mode = x_train["city"].mode()[0]
x_train["city"] = x_train["city"].fillna(city_mode)
x_test["city"] = x_test["city"].fillna(city_mode)

x_train["city"] = x_train["city"].str.strip().str.lower()
x_test["city"] = x_test["city"].str.strip().str.lower()

city_map = {"nyc": "new york"}
x_train["city"] = x_train["city"].replace(city_map)
x_test["city"] = x_test["city"].replace(city_map)

# %%
# --- Clean years_experience ---
median_exp = x_train["years_experience"].median()
x_train["years_experience"] = x_train["years_experience"].fillna(median_exp)
x_test["years_experience"] = x_test["years_experience"].fillna(median_exp)

invalid_train = x_train["years_experience"] > (x_train["age"] - 18)
x_train.loc[invalid_train, "years_experience"] = (x_train.loc[invalid_train, "age"] - 18).clip(lower=0)

invalid_test = x_test["years_experience"] > (x_test["age"] - 18)
x_test.loc[invalid_test, "years_experience"] = (x_test.loc[invalid_test, "age"] - 18).clip(lower=0)

# %%
# --- Clean remote_work ---
mode_remote = x_train["remote_work"].mode()[0]
x_train["remote_work"] = x_train["remote_work"].fillna(mode_remote)
x_test["remote_work"] = x_test["remote_work"].fillna(mode_remote)

# %%
# --- Clean performance_rating (NEW) ---
median_perf = x_train["performance_rating"].median()
x_train["performance_rating"] = x_train["performance_rating"].fillna(median_perf)
x_test["performance_rating"] = x_test["performance_rating"].fillna(median_perf)

# %%
# --- One-hot encode ---
x_train = pd.get_dummies(x_train, columns=["department", "city"], drop_first=True).astype(int)
x_test = pd.get_dummies(x_test, columns=["department", "city"], drop_first=True).astype(int)

# Align columns in case train/test have different categories present
x_train, x_test = x_train.align(x_test, join="left", axis=1, fill_value=0)

# %%
# --- Clean salary (target) ---
y_train = y_train.astype(str).str.replace(r'[^\d.]', '', regex=True).astype(float)
y_test = y_test.astype(str).str.replace(r'[^\d.]', '', regex=True).astype(float)

median_salary = y_train.median()
y_train[y_train > 200000] = median_salary
y_test[y_test > 200000] = median_salary

# %%
# --- Scale numeric columns (FIXED — transform only on test) ---
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
cols_to_scale = ["age", "years_experience", "performance_rating"]

x_train[cols_to_scale] = scaler.fit_transform(x_train[cols_to_scale])
x_test[cols_to_scale] = scaler.transform(x_test[cols_to_scale])   # fixed: transform, not fit_transform

# %%
# --- Train model ---
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

model = LinearRegression()
model.fit(x_train, y_train)
pred = model.predict(x_test)

print("MAE :", mean_absolute_error(y_test, pred))
print("R2 :", r2_score(y_test, pred))

MAE : 10964.785075959886
R2 : 0.3693532766342704


In [2]:
print(y_train.groupby(x_train["performance_rating"]).mean())

performance_rating
-1.627444    74303.383852
-0.549925    74953.673223
 0.527593    76577.167386
 1.605112    78946.150000
 2.682631    83837.546667
Name: salary, dtype: float64
